In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Reshape,
    LSTM,
    Dense,
    Dropout,
    TimeDistributed,
    Flatten
)

from tensorflow.keras.models import Model

In [ ]:
input_layer = Input(
    shape=(64, 64, 32)
)

x = Conv2D(
    32,
    (3,3),
    activation='relu',
    padding='same'
)(input_layer)

x = MaxPooling2D((2,2))(x)

x = Conv2D(
    64,
    (3,3),
    activation='relu',
    padding='same'
)(x)

x = MaxPooling2D((2,2))(x)

x = Reshape(
    (16, 16*64)
)(x)

x = LSTM(
    128,
    return_sequences=False
)(x)

x = Dropout(0.5)(x)

x = Dense(
    64,
    activation='relu'
)(x)

output_layer = Dense(
    2,
    activation='softmax'
)(x)

sogpcn_model = Model(
    inputs=input_layer,
    outputs=output_layer
)

sogpcn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

sogpcn_model.summary()

In [ ]:
history_sogpcn = sogpcn_model.fit(
    X_train,
    y_train,
    validation_data=(
        X_test,
        y_test
    ),
    epochs=15,
    batch_size=8
)

In [ ]:
plt.plot(
    history_sogpcn.history['accuracy']
)

plt.plot(
    history_sogpcn.history['val_accuracy']
)

plt.title(
    "SOGPCN Model Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend([
    'Train',
    'Validation'
])

plt.savefig(
    "/kaggle/working/sogpcn_accuracy.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.plot(
    history_sogpcn.history['loss']
)

plt.plot(
    history_sogpcn.history['val_loss']
)

plt.title(
    "SOGPCN Model Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend([
    'Train',
    'Validation'
])

plt.savefig(
    "/kaggle/working/sogpcn_loss.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
sample = X_test[0]

prediction = sogpcn_model.predict(
    sample[np.newaxis, ...]
)

predicted_class = np.argmax(
    prediction
)

print(prediction)

print(predicted_class)

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='inferno'
)

plt.title(
    f"SOGPCN Spectrogram | Predicted: {predicted_class}"
)

plt.colorbar()

plt.savefig(
    "/kaggle/working/sogpcn_spectrogram.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
grad_model = tf.keras.models.Model(
    [sogpcn_model.inputs],
    [
        sogpcn_model.get_layer(index=3).output,
        sogpcn_model.output
    ]
)

input_image = sample[np.newaxis, ...]

with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(
        input_image
    )

    class_idx = tf.argmax(
        predictions[0]
    )

    loss = predictions[:, class_idx]

grads = tape.gradient(
    loss,
    conv_outputs
)

pooled_grads = tf.reduce_mean(
    grads,
    axis=(0,1,2)
)

conv_outputs = conv_outputs[0]

heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

heatmap = tf.squeeze(heatmap)

heatmap = np.maximum(
    heatmap,
    0
)

heatmap /= np.max(
    heatmap
)

plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='gray'
)

plt.imshow(
    heatmap,
    cmap='jet',
    alpha=0.5
)

plt.title(
    "SOGPCN GradCAM"
)

plt.colorbar()

plt.savefig(
    "/kaggle/working/sogpcn_gradcam.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
real_importance = np.std(
    sample,
    axis=(0,1)
)

real_importance = (
    real_importance -
    np.min(real_importance)
)

real_importance = (
    real_importance /
    np.max(real_importance)
)

fig, ax = plt.subplots(figsize=(8,8))

mne.viz.plot_topomap(
    real_importance,
    info,
    cmap='jet',
    contours=6,
    axes=ax,
    show=False
)

plt.title(
    "SOGPCN EEG Topomap"
)

plt.savefig(
    "/kaggle/working/sogpcn_topomap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
predictions = sogpcn_model.predict(
    X_test
)

y_pred = np.argmax(
    predictions,
    axis=1
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

report = classification_report(
    y_test,
    y_pred,
    zero_division=0
)

print(report)